In [33]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import r2_score, mean_absolute_error
import shap


In [34]:
EXCEL_PATH     = "CRC-AV-33-22_Supplementary-Data.xlsx"
SHEET_NAME     = "Data"
TARGET_NAME    = "Ignition Delay"   # D6890 / IQT method (row 271 in spreadsheet)
N_TOP_FEATURES = 200                 # number of SHAP-selected features for model
MAX_MISSING    = 5                  # max allowed NaNs per column (out of 38 rows)
SEED           = 42
OUT_PATH       = "ignition_delay_results.png"

In [35]:
# style helpers

FIG_BG    = "#0f1117"
DARK_BG   = "#1a1d27"
AXIS_CLR  = "#9399b2"
TEXT_CLR  = "#cdd6f4"
TITLE_CLR = "#cdd6f4"
SPINE_CLR = "#363a4f"
 
def style_ax(ax):
    ax.set_facecolor(DARK_BG)
    ax.tick_params(colors=AXIS_CLR)
    for sp in ax.spines.values():
        sp.set_edgecolor(SPINE_CLR)
    ax.xaxis.label.set_color(TEXT_CLR)
    ax.yaxis.label.set_color(TEXT_CLR)

In [36]:
def load_data(excel_path, sheet_name, target_name):
    """
    The CRC spreadsheet is transposed:
        rows  = 397 fuel properties
        cols  = 9 metadata columns + 38 fuel samples
 
    We pivot to the ML convention (38 rows × N features) and extract the target.
    Duplicate property names (multiple Viscosity / Density rows etc.) are
    de-duplicated by appending __1, __2, … suffixes so no column is silently lost.
    """
    print(f"Loading {excel_path} …")
    df_raw = pd.read_excel(excel_path, sheet_name=sheet_name)
 
    # de-duplicate row (property) names before using them as column headers
    raw_names, seen, unique_names = df_raw.iloc[:, 0].tolist(), {}, []
    for name in raw_names:
        key = str(name)
        seen[key] = seen.get(key, 0)
        unique_names.append(key if seen[key] == 0 else f"{key}__{seen[key]}")
        seen[key] += 1
 
    # sample data starts at column index 9
    df_T         = df_raw.iloc[:, 9:].T.copy()
    df_T.columns = unique_names
    df_T.index   = range(1, 39)
    df_T         = df_T.apply(pd.to_numeric, errors="coerce")
 
    if target_name not in df_T.columns:
        candidates = [c for c in df_T.columns if "ignition" in c.lower()]
        raise KeyError(f"'{target_name}' not found. Candidates: {candidates}")
 
    y = df_T[target_name].values.astype(float)
    print(f"Target '{target_name}': n={len(y)}, "
          f"range [{y.min():.3f}, {y.max():.3f}] ms, mean={y.mean():.3f} ms")
    return df_T, y

In [37]:
DROPPED = {
    "Ignition Delay", "Ignition delay", "Ignition delay__1",   # both ID measurements
    "Derived Cetane Number", "Indicated Cetane Number",         # functions of ID
    "Avg. Air Temperature", "Temperature",                      # IQT instrument conditions
    "Chamber pressure", "Injection pressure",                   # IQT instrument conditions
    "Property \\ Fuel Type", "ASTM Code", "Criteria",          # metadata
}
 
def build_feature_matrix(df_full, target_name, max_missing=MAX_MISSING):
    drop     = DROPPED | {target_name}
    all_nan  = {c for c in df_full.columns if df_full[c].isna().all()}
    keep     = [c for c in df_full.columns
                if c not in drop
                and c not in all_nan
                and df_full[c].isna().sum() <= max_missing]
 
    X = df_full[keep].copy()
    X = X.fillna(X.median())                      # fill the few residual NaNs
    X = X.loc[:, X.var() > 1e-10]                 # drop zero-variance columns
    print(f"Feature matrix: {X.shape}")
    return X


In [38]:
def select_features_shap(X, y, n=N_TOP_FEATURES):
    """
    Fit a Random Forest on all features, compute SHAP values, rank by
    mean |SHAP|, and return the top-n feature names.
 
    The RF here is used purely as the SHAP vehicle — it is fit on the full
    dataset (no CV), which is fine because SHAP is used only for ranking,
    not for final prediction accuracy.
    """
    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(X)
 
    print("Fitting Random Forest for SHAP …")
    rf = RandomForestRegressor(n_estimators=500, max_depth=6,
                                min_samples_leaf=2, max_features="sqrt",
                                random_state=SEED)
    rf.fit(X_scaled, y)
 
    print("Computing SHAP values …")
    explainer   = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(X_scaled)          # shape (38, n_features)
    shap_imp    = pd.Series(np.abs(shap_values).mean(axis=0),
                            index=X.columns).sort_values(ascending=False)
 
    top_features = shap_imp.head(n).index.tolist()
    print(f"\nTop {n} features by SHAP importance:")
    for i, feat in enumerate(top_features):
        print(f"  {i+1:2d}. {feat:<42s}  mean|SHAP|={shap_imp[feat]:.5f}")
 
    return top_features, shap_values, shap_imp, X_scaled

In [39]:
def train_ridge_loo(X_top, y):
    """
    Fit Ridge regression on the top features using Leave-One-Out CV.
    Returns per-sample LOO predictions and the model fit on the full dataset.
    """
    scaler   = StandardScaler()
    X_s      = scaler.fit_transform(X_top)
    loo      = LeaveOneOut()
    loo_preds = np.zeros(len(y))
 
    for tr, te in loo.split(X_s):
        model = Ridge(alpha=1.0)
        model.fit(X_s[tr], y[tr])
        loo_preds[te] = model.predict(X_s[te])
 
    mae = mean_absolute_error(y, loo_preds)
    r2  = r2_score(y, loo_preds)
    print(f"\nRidge LOO-CV  →  MAE = {mae:.4f} ms   R² = {r2:.4f}")
 
    # also fit on full data (for coefficient inspection if needed)
    model_full = Ridge(alpha=1.0).fit(X_s, y)
    return loo_preds, mae, r2, model_full, scaler


In [40]:
def plot_results(y, loo_preds, mae, r2,
                 shap_values_full, X_scaled_full, X,
                 top_features, shap_imp, out_path):
    """
    Panel A — LOO parity plot (predicted vs. measured)
    Panel B — SHAP beeswarm for the top-10 selected features
    Panel C — Sorted prediction comparison (trend sanity check)
    """
    resid = y - loo_preds
    rng   = np.random.default_rng(SEED)
 
    # column indices for the top features inside the full X_scaled matrix
    top_idx = [list(X.columns).index(f) for f in top_features]
 
    fig = plt.figure(figsize=(14, 18))  # taller figure
    fig.patch.set_facecolor(FIG_BG)
    gs  = gridspec.GridSpec(3, 1, figure=fig, hspace=0.45)
 
    # ── A: Parity plot ────────────────────────────────────────────────────────
    ax = fig.add_subplot(gs[0]); style_ax(ax)
    vmin, vmax = y.min() - 0.15, y.max() + 0.15
    ax.plot([vmin, vmax], [vmin, vmax], "--", color="#f9a825", lw=1.5,
            label="Perfect fit")
    sc = ax.scatter(y, loo_preds, c=np.abs(resid), cmap="RdYlGn_r",
                    s=75, edgecolors="#cdd6f4", lw=0.6, vmin=0, vmax=0.5,
                    zorder=3)
    cb = plt.colorbar(sc, ax=ax)
    cb.set_label("|Residual| (ms)", color=TEXT_CLR, fontsize=9)
    cb.ax.yaxis.label.set_color(TEXT_CLR)
    cb.ax.tick_params(colors=AXIS_CLR)
    # label each point with its sample number
    for i in range(len(y)):
        ax.annotate(str(i + 1), (y[i], loo_preds[i]),
                    fontsize=6, color="#cdd6f4", alpha=0.65,
                    xytext=(3, 3), textcoords="offset points")
    ax.set_xlabel("Measured Ignition Delay (ms)", fontsize=11)
    ax.set_ylabel("LOO-CV Predicted (ms)",        fontsize=11)
    ax.set_title(f"A  Ridge Regression — LOO Parity\n"
                 f"MAE = {mae:.3f} ms   R² = {r2:.3f}   n = {len(y)}",
                 color=TITLE_CLR, fontsize=12, fontweight="bold", loc="left")
    ax.legend(fontsize=9, facecolor="#2a2d3a", labelcolor=TEXT_CLR)
 
    # ── B: SHAP beeswarm ──────────────────────────────────────────────────────
    ax2 = fig.add_subplot(gs[1]); style_ax(ax2)
    # plot bottom-to-top so the most important feature is at the top
    for yi, fi in enumerate(top_idx[::-1]):
        sv      = shap_values_full[:, fi]
        fv_norm = (X_scaled_full[:, fi] - X_scaled_full[:, fi].min()) / \
                  (np.ptp(X_scaled_full[:, fi]) + 1e-8)
        jitter  = rng.uniform(-0.38, 0.38, len(sv))
        ax2.scatter(sv, yi + jitter, c=fv_norm, cmap="RdBu_r",
                    s=35, alpha=0.85, vmin=0, vmax=1)
 
    ax2.set_yticks(range(len(top_features)))
    ax2.set_yticklabels([f[:32] for f in top_features[::-1]],
                         fontsize=8.5, color=TEXT_CLR)
    ax2.axvline(0, color="#f9a825", lw=1.2, ls="--", alpha=0.7)
    ax2.set_xlabel("SHAP value  (impact on predicted ID)", fontsize=10)
    ax2.set_title("B  SHAP Beeswarm\n"
                  "Blue = low feature value   Red = high",
                  color=TITLE_CLR, fontsize=12, fontweight="bold", loc="left")
 
    sm  = plt.cm.ScalarMappable(cmap="RdBu_r", norm=plt.Normalize(0, 1))
    cb2 = plt.colorbar(sm, ax=ax2, orientation="vertical",
                       fraction=0.03, pad=0.02)
    cb2.set_label("Feature value\n(normalised)", color=TEXT_CLR, fontsize=8)
    cb2.ax.yaxis.label.set_color(TEXT_CLR)
    cb2.ax.tick_params(colors=AXIS_CLR, labelsize=7)
 
    # ── C: Sorted prediction comparison ──────────────────────────────────────
    ax3 = fig.add_subplot(gs[2]); style_ax(ax3)
    sort_idx = np.argsort(y)
    x_pos    = np.arange(len(y))
 
    ax3.plot(x_pos, y[sort_idx], "o-", color="#f9a825",
             lw=2, ms=6, label="Measured", alpha=0.95, zorder=3)
    ax3.plot(x_pos, loo_preds[sort_idx], "s--", color="#5b8dee",
             lw=1.8, ms=5, label="Ridge LOO", alpha=0.85)
 
    # shade the residual gap
    ax3.fill_between(x_pos,
                     y[sort_idx], loo_preds[sort_idx],
                     alpha=0.12, color="#5b8dee")
 
    ax3.set_xlabel("Sample index (sorted by measured ID)", fontsize=10)
    ax3.set_ylabel("Ignition Delay (ms)",                  fontsize=10)
    ax3.set_title("C  Sorted Prediction Comparison\n"
                  "Shaded area = prediction error",
                  color=TITLE_CLR, fontsize=12, fontweight="bold", loc="left")
    ax3.legend(fontsize=9, facecolor="#2a2d3a", labelcolor=TEXT_CLR)
 
    fig.suptitle(
        "Ignition Delay Prediction  ·  CRC Aviation Fuel Dataset (n=38, D6890/IQT)\n"
        f"SHAP feature selection (top {N_TOP_FEATURES})  →  Ridge regression  →  LOO-CV",
        color="#89b4fa", fontsize=13, fontweight="bold", y=1.02)
 
    plt.savefig(out_path, dpi=150, bbox_inches="tight",
                facecolor=FIG_BG, edgecolor="none")
    plt.close()
    print(f"\nFigure saved → {out_path}")

In [41]:
df_full, y = load_data(EXCEL_PATH, SHEET_NAME, TARGET_NAME)

# 2. feature matrix
X = build_feature_matrix(df_full, TARGET_NAME, MAX_MISSING)

# 3. SHAP selection  (RF fit on full data, used only for ranking)
top_features, shap_values, shap_imp, X_scaled = select_features_shap(X, y, N_TOP_FEATURES)

# 4. Ridge regression on top features with LOO-CV
X_top = X[top_features].copy()
loo_preds, mae, r2, _, _ = train_ridge_loo(X_top, y)

# 5. plot
plot_results(y, loo_preds, mae, r2,
                shap_values, X_scaled, X,
                top_features, shap_imp, OUT_PATH)

print("\n=== DONE ===")
print(f"  Top features : {top_features}")
print(f"  LOO MAE      : {mae:.4f} ms")
print(f"  LOO R²       : {r2:.4f}")

Loading CRC-AV-33-22_Supplementary-Data.xlsx …
Target 'Ignition Delay': n=38, range [3.970, 5.590] ms, mean=4.812 ms
Feature matrix: (38, 298)
Fitting Random Forest for SHAP …
Computing SHAP values …

Top 200 features by SHAP importance:
   1. n-alkane-C13                                mean|SHAP|=0.01658
   2. iso-alkane-C13                              mean|SHAP|=0.01380
   3. iso-alkane-C14                              mean|SHAP|=0.01154
   4. Net Heat of Combustion_value                mean|SHAP|=0.01117
   5. Smoke point_value                           mean|SHAP|=0.00980
   6. Hydrogen content                            mean|SHAP|=0.00971
   7. n-alkane-C14                                mean|SHAP|=0.00959
   8. n-alkane-C12                                mean|SHAP|=0.00874
   9. Monocycloalkane-C09                         mean|SHAP|=0.00604
  10. Dicycloalkane-C10                           mean|SHAP|=0.00578
  11. Naphthalenes_value                          mean|SHAP|=0.00575
  1